# RAS metric_fast — version 2.0

This notebook implements the Kaggle scoring metric for the INFORMS RAS 2026 Problem Solving Competition.

**Metric v2.0 construction:** this notebook keeps the Kaggle/official-input wrapper from `metric_fast_v1_1`, and updates the validation and scoring core to match `fast_validator_v2_0.py`.

## Update log to v2.0

1. **Use v2.0 rule parameters and defaults.**
   - Minimum block-volume defaults are aligned with the v2.0 public validator: 350 / 700 / 1050 cars for short / medium / long blocks.
   - The metric still loads official `setting.csv` values when available, and assigns `demand_multiplier` by official case ID.
   - Participant-submitted `inputs.settings` are ignored.
   - Official `node.csv`, `link.csv`, `demand.csv`, and optional `setting.csv` are loaded by case ID.
   - Submitted rows only need an `outputs` section.

2. **Update C2 classification block types.**
   - C2 now counts classification blocks with block types `{manifest, coal, grain}`.

4. **Add check `1b_no_subtour`.**
   - A blocking sequence may not revisit the same yard.

5. **Add check `9_direct_block_rule`.**
   - Intermodal and Automobile commodities must use a single direct OD block.
   - Their `blocking_sequence` may contain only one `block_id`.

6. **Keep demand transported-volume consistency as `9b_demand_volume_consistency`.**
   - Under-service is allowed and penalized through stress-score unserved car-miles.
   - Non-positive submitted sequence rows, unknown demand references, OD mismatches, and over-service remain validation errors.
   - Served demand is measured by submitted positive volume, not by demand-key appearance.

7. **Update interchange cost to the v2.0 two-endpoints Class-I rule.**
   - For each used block, compare the railroad IDs of the block origin yard and destination yard.
   - If both endpoints are recognized Class-I railroads and they are different, charge one interchange cost to the block volume.
   - Intermediate physical-path nodes are ignored for interchange-cost calculation.
   - CSXT is normalized to CSX.

8. **Keep the v1.1 consistency fixes.**
   - Zero/negative blocking-sequence volume is rejected.
   - Transported volume cannot exceed the corresponding official input demand volume.
   - Demand origin/destination/type references are checked against official input demands.


## Step 1 — imports and v2.0 constants

In [ ]:
import json
import math
from pathlib import Path

import pandas as pd
import numpy as np

class ParticipantVisibleError(Exception):
    pass

# Keep these defaults aligned with DATASET_README / setting.csv and fast_validator_v2_0.py.
# Official setting.csv values override these defaults when available.
# demand_multiplier is assigned by official case ID through DEMAND_MULTIPLIER_BY_ID.
DEFAULT_SETTINGS = {
    "min_block_vol_short(<100mi)": 350,
    "min_block_vol_med(100-500mi)": 700,
    "min_block_vol_long(>500mi)": 1050,
    "max_circuitous_ratio": 1.3,
    "operating_cycle": 70,
    "block_fixed_cost": 2500.0,
    "transport_cost_coefficient": 1.0,
    "interchange_cost": 100.0,
    "stress_penalty_M": 5.0,
    "demand_multiplier": 1.0,
}

# Class-I railroad labels used by the v2.0 two-endpoints interchange rule.
# CSXT is normalized to CSX in _class_i_rr().
CLASS_I_RAILROADS = {"BNSF", "CN", "CSX", "UP", "NS", "CPKC"}

# Official scenario demand multipliers by Kaggle row/case ID.
# IDs 0-2, 3-5, and 6-8 correspond to the three demand scenarios
# for each network scale. Participant-submitted demand_multiplier is ignored.
DEMAND_MULTIPLIER_BY_ID = {
    0: 0.5,
    1: 1.0,
    2: 2.0,
    3: 0.5,
    4: 1.0,
    5: 2.0,
    6: 0.5,
    7: 1.0,
    8: 2.0,
}

# ---------------------------------------------------------------------
# Official input data location
# ---------------------------------------------------------------------
# Local mode: set this to the parent directory that contains datasets/l1,
# datasets/l2, and datasets/l3. Leave it blank when using Kaggle mode only.

LOCAL_OFFICIAL_DATA_ROOT = Path("..")

# Kaggle mode: official competition data are expected under this root.
# The actual CSV files are read from:
#   KAGGLE_COMPETITION_ROOT / "datasets" / {l1,l2,l3} / {node,link,demand,setting}.csv
KAGGLE_COMPETITION_ROOT = Path("/kaggle/input/competitions/informs-ras-2026-problem-solving-competition/")

# Case-ID to network-layer mapping:
#   IDs 0,1,2 -> l1
#   IDs 3,4,5 -> l2
#   IDs 6,7,8 -> l3
CASE_LAYER_BY_ID = {
    0: "l1",
    1: "l1",
    2: "l1",
    3: "l2",
    4: "l2",
    5: "l2",
    6: "l3",
    7: "l3",
    8: "l3",
}

# C2 counts only classification blocks under the RAS v2.0 interpretation.
# Intermodal / Automobile use direct blocks and are not reclassified.
CLASSIFICATION_BLOCK_TYPES = {"manifest", "coal", "grain"}


def _is_direct_only(commodity_label) -> bool:
    """Return True for commodity types that must use a single direct OD block."""
    s = str(commodity_label or "").strip().lower()
    return (
        "intermodal" in s
        or "automobile" in s
    )


## Step 2 — v2.0 validator functions

In [ ]:
def _merge_settings(raw_settings: dict | None = None) -> dict:
    """Return metric-owned settings, optionally overridden by official setting.csv.

    Participant-submitted inputs.settings are never used.  The only accepted
    external settings are those loaded from the official dataset directory by
    _load_official_settings_for_case().  demand_multiplier is still assigned by
    case ID through DEMAND_MULTIPLIER_BY_ID.
    """
    settings = DEFAULT_SETTINGS.copy()
    if isinstance(raw_settings, dict):
        for k, v in raw_settings.items():
            if k in settings and k != "demand_multiplier":
                settings[k] = v

    for k, default in DEFAULT_SETTINGS.items():
        try:
            settings[k] = float(settings[k])
        except (TypeError, ValueError):
            settings[k] = default
    return settings


def _case_layer_from_id(case_id) -> str:
    """Map Kaggle row/case ID to official network layer l1/l2/l3."""
    try:
        cid = int(case_id)
    except Exception:
        raise ParticipantVisibleError(f"Invalid case ID {case_id!r}: cannot determine official input layer.")
    if cid not in CASE_LAYER_BY_ID:
        raise ParticipantVisibleError(f"Invalid case ID {case_id!r}: no official input layer is defined.")
    return CASE_LAYER_BY_ID[cid]


def _candidate_official_data_roots() -> list[Path]:
    """Return candidate roots that may contain the official datasets directory."""
    candidates: list[Path] = []

    # 1) User-specified local root. Leave LOCAL_OFFICIAL_DATA_ROOT blank to skip.
    local_root_text = str(LOCAL_OFFICIAL_DATA_ROOT).strip()
    if local_root_text not in {"", "."}:
        candidates.append(Path(LOCAL_OFFICIAL_DATA_ROOT))

    # 2) Kaggle official competition input root.
    candidates.append(KAGGLE_COMPETITION_ROOT)

    # 3) Convenient local fallbacks, useful when running the notebook beside datasets/.
    candidates.extend([Path.cwd(), Path.cwd().parent])

    # Deduplicate while preserving order.
    out: list[Path] = []
    seen: set[str] = set()
    for p in candidates:
        try:
            key = str(p.resolve())
        except Exception:
            key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def _datasets_dir_from_root(root: Path) -> Path:
    """Normalize a root to the directory containing l1/l2/l3."""
    root = Path(root)
    return root if root.name.lower() == "datasets" else root / "datasets"


def _official_case_input_dir(case_id) -> Path:
    """Return the official input directory for one case ID.

    Expected structure:
        <root>/datasets/l1/node.csv
        <root>/datasets/l1/link.csv
        <root>/datasets/l1/demand.csv
        <root>/datasets/l1/setting.csv  (optional)

    IDs 0-2 use l1, IDs 3-5 use l2, and IDs 6-8 use l3.
    """
    layer = _case_layer_from_id(case_id)
    tried: list[str] = []
    for root in _candidate_official_data_roots():
        d = _datasets_dir_from_root(root) / layer
        tried.append(str(d))
        if d.exists() and d.is_dir():
            return d
    raise ParticipantVisibleError(
        f"Official input directory for case ID {case_id!r} ({layer}) was not found. "
        "Set LOCAL_OFFICIAL_DATA_ROOT to the parent directory containing datasets/l1,l2,l3, "
        "or ensure the Kaggle competition data are mounted. Tried: " + "; ".join(tried)
    )


def _read_required_csv(path: Path, label: str) -> pd.DataFrame:
    """Read a required official CSV file and normalize column names.

    The released v2 CSV files are the source of truth.  This helper normalizes
    minor CSV-export differences before the validator applies its internal
    schema:
      - strips BOM and surrounding spaces from column names;
      - lowercases column names and replaces spaces with underscores;
      - drops fully empty rows.
    """
    path = Path(path)
    if not path.exists():
        raise ParticipantVisibleError(f"Official {label} file was not found: {path}")
    try:
        df = pd.read_csv(path, sep=None, engine="python")
    except Exception as e:
        raise ParticipantVisibleError(f"Failed to read official {label} file {path}: {e}")

    df.columns = (
        df.columns.astype(str)
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return df.dropna(how="all").copy()


def _parse_official_setting_csv(setting_path: Path) -> dict:
    """Read optional official setting.csv into a dict.

    This parser accepts common two-column layouts such as
    setting/value, parameter/value, key/value, or simply the first two columns.
    If setting.csv is absent, DEFAULT_SETTINGS are used.
    """
    setting_path = Path(setting_path)
    if not setting_path.exists():
        return {}
    try:
        df = pd.read_csv(setting_path)
    except Exception as e:
        raise ParticipantVisibleError(f"Failed to read official setting file {setting_path}: {e}")
    if df.empty or len(df.columns) < 2:
        return {}

    lower_to_col = {str(c).strip().lower(): c for c in df.columns}
    key_col = None
    for name in ["setting", "settings", "parameter", "param", "key", "name"]:
        if name in lower_to_col:
            key_col = lower_to_col[name]
            break
    if key_col is None:
        key_col = df.columns[0]

    value_col = None
    for name in ["value", "val"]:
        if name in lower_to_col:
            value_col = lower_to_col[name]
            break
    if value_col is None:
        value_col = df.columns[1]

    out = {}
    for _, r in df.iterrows():
        k = str(r[key_col]).strip()
        if k and k.lower() != "nan":
            out[k] = r[value_col]
    return out


def _load_official_inputs_for_case(case_id) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, dict, Path]:
    """Load official node/link/demand/setting files for a case ID.

    Submitted JSON inputs are intentionally ignored.  The participant payload
    should contain only outputs; all official inputs are loaded here.

    Internal schema used by the validator:
      demand_id    -> commodity_id
      block_type   -> commodity_type
      dest_yard_id -> dest_yard_id  (kept as the internal destination field)
    """
    d = _official_case_input_dir(case_id)
    nodes_df = _read_required_csv(d / "node.csv", "node.csv")
    links_df = _read_required_csv(d / "link.csv", "link.csv")
    demands_df = _read_required_csv(d / "demand.csv", "demand.csv")
    official_settings = _parse_official_setting_csv(d / "setting.csv")

    # Normalize v2 demand.csv column names to the validator's internal schema.
    # Keep "dest_yard_id" as the internal field because participant outputs and
    # the downstream validation logic already use this name.
    demands_df = demands_df.rename(columns={
        "demand_id": "commodity_id",
        "destination_yard_id": "dest_yard_id",
        "block_type": "commodity_type",
    })

    required_demand_cols = [
        "commodity_id",
        "commodity_type",
        "origin_yard_id",
        "dest_yard_id",
        "volume",
    ]
    missing_cols = [c for c in required_demand_cols if c not in demands_df.columns]
    if missing_cols:
        raise ParticipantVisibleError(
            "Official demand.csv is missing required columns after normalization: "
            f"{missing_cols}. Available columns are: {list(demands_df.columns)}"
        )

    demands_df = demands_df[demands_df["commodity_id"].notna()].copy()
    demands_df["commodity_id"] = pd.to_numeric(demands_df["commodity_id"], errors="raise").astype(int)
    demands_df["volume"] = pd.to_numeric(demands_df["volume"], errors="raise")

    return nodes_df, links_df, demands_df, official_settings, d


def _settings_for_case_id(case_id=None) -> dict:
    """Return official metric settings, with demand_multiplier fixed by case ID."""
    official_settings = {}
    if case_id is not None:
        # setting.csv is read only from the official input directory, never from the submission JSON.
        try:
            _, _, _, official_settings, _ = _load_official_inputs_for_case(case_id)
        except ParticipantVisibleError:
            raise
        except Exception:
            official_settings = {}

    settings = _merge_settings(official_settings)
    if case_id is not None:
        try:
            cid = int(case_id)
        except Exception:
            raise ParticipantVisibleError(f"Invalid case ID {case_id!r}: cannot determine demand_multiplier.")
        if cid not in DEMAND_MULTIPLIER_BY_ID:
            raise ParticipantVisibleError(
                f"Invalid case ID {case_id!r}: no official demand_multiplier is defined."
            )
        settings["demand_multiplier"] = float(DEMAND_MULTIPLIER_BY_ID[cid])
    return settings


def _empty_input_reason(data: dict) -> str | None:
    """Return a visible invalid reason if required input lists are missing/empty."""
    inputs = data.get("inputs", None)
    if not isinstance(inputs, dict):
        return "missing or invalid inputs section"
    for key in ["nodes", "links", "demands"]:
        value = inputs.get(key, None)
        if value is None:
            return f"inputs.{key} is missing"
        if not isinstance(value, list):
            return f"inputs.{key} must be a list"
        if len(value) == 0:
            return f"inputs.{key} is empty"
    return None


def _check_nonempty_inputs(data: dict) -> None:
    """Reject non-empty submissions that try to score with empty official-like inputs."""
    reason = _empty_input_reason(data)
    if reason is not None:
        raise ParticipantVisibleError(
            "Invalid submission: " + reason + ". "
            "Use the official released case data; empty input sections cannot be scored."
        )


def _min_vol_scalar(dist: float, s: dict) -> float:
    """RAS threshold rule used by the strict validator: <100, 100--500, >500."""
    if dist < 100:
        return float(s["min_block_vol_short(<100mi)"])
    if dist <= 500:
        return float(s["min_block_vol_med(100-500mi)"])
    return float(s["min_block_vol_long(>500mi)"])


def _parse_arrow_ints(value) -> list[int]:
    if value is None:
        return []
    try:
        if pd.isna(value):
            return []
    except Exception:
        pass
    return [int(x) for x in str(value).split(" -> ") if str(x).strip()]


def _distance_from_link_path(path_value, link_len: dict[int, float]) -> float:
    if path_value is None:
        return 0.0
    try:
        if pd.isna(path_value):
            return 0.0
    except Exception:
        pass

    import re
    tokens = re.findall(r"-?\d+", str(path_value))
    total = 0.0
    for t in tokens:
        lid = int(t)
        total += float(link_len.get(lid, link_len.get(abs(lid), 0.0)))
    return total


def _shortest_path_cache_scipy(links_df: pd.DataFrame, od_pairs: set[tuple[int, int]]) -> dict[tuple[int, int], float]:
    """Compute shortest-path distances using SciPy sparse Dijkstra.

    The reference validator treats physical links as bidirectional. If the input
    already contains both directions, duplicate directed arcs must be coalesced
    before building the CSR matrix, otherwise SciPy will sum duplicate weights.
    For duplicate arcs, keep the minimum length.
    """
    from scipy.sparse import csr_matrix
    from scipy.sparse.csgraph import dijkstra

    out: dict[tuple[int, int], float] = {}
    if not od_pairs:
        return out

    if links_df.empty or "from_node_id" not in links_df.columns or "to_node_id" not in links_df.columns:
        for u, v in od_pairs:
            out[(int(u), int(v))] = np.nan
        return out

    node_ids = pd.Index(pd.unique(pd.concat([
        links_df["from_node_id"].astype(int),
        links_df["to_node_id"].astype(int),
    ], ignore_index=True)))
    node_to_i = {int(n): i for i, n in enumerate(node_ids)}

    edge_w: dict[tuple[int, int], float] = {}
    for _, l in links_df.iterrows():
        u = int(l["from_node_id"])
        v = int(l["to_node_id"])
        w = float(l["length"])
        iu, iv = node_to_i[u], node_to_i[v]
        for a, b in ((iu, iv), (iv, iu)):
            old_w = edge_w.get((a, b))
            if old_w is None or w < old_w:
                edge_w[(a, b)] = w

    if not edge_w:
        for u, v in od_pairs:
            out[(int(u), int(v))] = np.nan
        return out

    rows = [k[0] for k in edge_w.keys()]
    cols = [k[1] for k in edge_w.keys()]
    data = list(edge_w.values())
    mat = csr_matrix((data, (rows, cols)), shape=(len(node_ids), len(node_ids)))

    by_origin: dict[int, set[int]] = {}
    for u, v in od_pairs:
        by_origin.setdefault(int(u), set()).add(int(v))

    for u, dests in by_origin.items():
        if u not in node_to_i:
            for v in dests:
                out[(u, v)] = np.nan
            continue
        dist = dijkstra(mat, directed=True, indices=node_to_i[u], return_predecessors=False)
        for v in dests:
            out[(u, v)] = float(dist[node_to_i[v]]) if v in node_to_i and np.isfinite(dist[node_to_i[v]]) else np.nan
    return out


def _shortest_path_cache(links_df: pd.DataFrame, od_pairs: set[tuple[int, int]]) -> tuple[dict[tuple[int, int], float], str]:
    """Compute shortest-path distances from the physical network using SciPy."""
    return _shortest_path_cache_scipy(links_df, od_pairs), "scipy_sparse_dijkstra"


def _candidate_od_matrix_paths() -> list[Path]:
    """Return likely locations of od_distance_matrix.csv in local/Kaggle environments."""
    candidates: list[Path] = []

    local_candidates = [
        Path.cwd() / "od_distance_matrix.csv",          # same folder as this notebook when run from scoring/
        Path("od_distance_matrix.csv"),                 # relative current folder
        Path("..") / "od_distance_matrix.csv",          # release root
        Path("..") / "datasets" / "od_distance_matrix.csv",
    ]

    for p in local_candidates:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if rp.exists() and rp not in candidates:
            candidates.append(rp)

    # for Kaggle
    kaggle_input = Path("/kaggle/input/competitions/informs-ras-2026-problem-solving-competition/")
    if kaggle_input.exists():
        try:
            for p in kaggle_input.glob("**/od_distance_matrix.csv"):
                try:
                    rp = p.resolve()
                except Exception:
                    rp = p
                if rp.exists() and rp not in candidates:
                    candidates.append(rp)
        except Exception:
            pass

    return candidates


def _resolve_od_matrix_path(od_distance_matrix_path: str | Path | None = None) -> Path | None:
    """Resolve an explicit or auto-discovered OD distance matrix path."""
    if od_distance_matrix_path is not None:
        p = Path(od_distance_matrix_path)
        return p if p.exists() else None
    for p in _candidate_od_matrix_paths():
        if p.exists():
            return p
    return None


def _load_od_distances_from_matrix(
    blocks_df: pd.DataFrame,
    demands_df: pd.DataFrame,
    od_distance_matrix_path: str | Path | None = None,
    verbose: bool = False,
) -> tuple[dict[tuple[int, int], float], dict]:
    """Load only required OD pairs from od_distance_matrix.csv using DuckDB.

    This mirrors validator_0517.py: construct required block/demand OD pairs,
    perform an INNER JOIN against the large matrix, and store both directions in
    memory. If the file is absent or DuckDB fails, return an empty dictionary so
    callers can fall back to SciPy.
    """
    od_distances: dict[tuple[int, int], float] = {}
    meta = {
        "od_matrix_path": None,
        "od_matrix_found": False,
        "od_matrix_loaded_pairs": 0,
        "od_matrix_error": None,
    }

    dist_file = _resolve_od_matrix_path(od_distance_matrix_path)

    if dist_file is None:
        return od_distances, meta

    meta["od_matrix_path"] = str(dist_file)
    meta["od_matrix_found"] = True

    needed_ods: set[tuple[int, int]] = set()
    if len(blocks_df) and {"from_yard_id", "to_yard_id"}.issubset(blocks_df.columns):
        for _, b in blocks_df.reset_index().iterrows():
            try:
                u, v = int(b["from_yard_id"]), int(b["to_yard_id"])
                needed_ods.add((u, v)); needed_ods.add((v, u))
            except Exception:
                pass
    if len(demands_df) and {"origin_yard_id", "dest_yard_id"}.issubset(demands_df.columns):
        for _, d in demands_df.iterrows():
            try:
                u, v = int(d["origin_yard_id"]), int(d["dest_yard_id"])
                needed_ods.add((u, v)); needed_ods.add((v, u))
            except Exception:
                pass

    if not needed_ods:
        return od_distances, meta

    try:
        import duckdb

        needed_df = (
            pd.DataFrame(list(needed_ods), columns=["u", "v"])
            .drop_duplicates()
            .sort_values(["u", "v"])
            .reset_index(drop=True)
        )
        con = duckdb.connect(database=":memory:")
        try:
            con.execute("PRAGMA enable_progress_bar;")
        except Exception:
            pass
        con.register("needed_df", needed_df)
        abs_dist_file = str(Path(dist_file).resolve()).replace("'", "''")
        query = f"""
            SELECT c.from_yard_id, c.to_yard_id, c.min_distance_mile
            FROM read_csv_auto('{abs_dist_file}') AS c
            INNER JOIN needed_df AS n
                ON c.from_yard_id = n.u AND c.to_yard_id = n.v
            ORDER BY from_yard_id, to_yard_id
        """
        if verbose:
            print(f"Discovered {dist_file}. Accelerating shortest paths with DuckDB...")
            print("  Executing DuckDB query over od_distance_matrix.csv...")
        results = con.execute(query).fetchall()
        con.close()

        for u, v, dist in results:
            od_distances[(int(u), int(v))] = float(dist)

        meta["od_matrix_loaded_pairs"] = len(od_distances)
        if verbose:
            print(f"  Loaded {meta['od_matrix_loaded_pairs']} required OD pairs into memory.")
    except Exception as e:
        meta["od_matrix_error"] = str(e)
        if verbose:
            print(f"  Warning: OD matrix load failed ({e}); falling back to SciPy for missing pairs.")

    return od_distances, meta


def _is_valid_sp_distance(u: int, v: int, dist) -> bool:
    """A shortest-path distance is valid if finite and positive, except same-yard OD may be zero."""
    try:
        d = float(dist)
    except (TypeError, ValueError):
        return False
    if pd.isna(d) or not np.isfinite(d):
        return False
    if d < 0:
        return False
    if d == 0 and int(u) != int(v):
        return False
    return True

def _combined_shortest_path_cache(
    links_df: pd.DataFrame,
    od_pairs: set[tuple[int, int]],
    blocks_df: pd.DataFrame,
    demands_df: pd.DataFrame,
    od_distance_matrix_path: str | Path | None = None,
    verbose: bool = False,
) -> tuple[dict[tuple[int, int], float], str, dict]:
    """Use one shortest-path convention everywhere: OD matrix first, SciPy physical-network fallback second.

    Pairs that cannot be obtained from either source remain unresolved and are reported in metadata.
    """
    od_matrix, meta = _load_od_distances_from_matrix(
        blocks_df=blocks_df,
        demands_df=demands_df,
        od_distance_matrix_path=od_distance_matrix_path,
        verbose=verbose,
    )

    requested_pairs = {(int(u), int(v)) for u, v in od_pairs}
    sp_cache: dict[tuple[int, int], float] = {}

    from_matrix = 0
    invalid_matrix = 0
    pairs_for_scipy: set[tuple[int, int]] = set()

    for u, v in requested_pairs:
        if (u, v) in od_matrix and _is_valid_sp_distance(u, v, od_matrix[(u, v)]):
            sp_cache[(u, v)] = float(od_matrix[(u, v)])
            from_matrix += 1
        else:
            if (u, v) in od_matrix:
                invalid_matrix += 1
            pairs_for_scipy.add((u, v))

    scipy_engine_used = False
    from_scipy = 0
    unresolved_after_scipy = 0
    if pairs_for_scipy:
        scipy_cache = _shortest_path_cache_scipy(links_df, pairs_for_scipy)
        scipy_engine_used = True
        for u, v in pairs_for_scipy:
            dist = scipy_cache.get((u, v), np.nan)
            if _is_valid_sp_distance(u, v, dist):
                sp_cache[(u, v)] = float(dist)
                from_scipy += 1
            else:
                unresolved_after_scipy += 1

    if meta.get("od_matrix_found") and meta.get("od_matrix_loaded_pairs", 0) > 0:
        engine = "duckdb_od_matrix"
        if scipy_engine_used:
            engine += "+scipy_sparse_dijkstra_fallback"
    else:
        engine = "scipy_sparse_dijkstra"

    meta["requested_shortest_path_pairs"] = len(requested_pairs)
    meta["shortest_path_pairs_from_matrix"] = from_matrix
    meta["shortest_path_pairs_from_scipy"] = from_scipy
    meta["shortest_path_pairs_unresolved"] = unresolved_after_scipy
    meta["invalid_matrix_pairs_recomputed_by_scipy"] = invalid_matrix
    return sp_cache, engine, meta

def _df_with_columns(records, columns: list[str]) -> pd.DataFrame:
    """Create a DataFrame and ensure required columns exist."""
    df = pd.DataFrame(records or [])
    for c in columns:
        if c not in df.columns:
            df[c] = pd.Series(dtype="object")
    return df


def fast_validate_payload(
    data: dict,
    case_id=None,
    od_distance_matrix_path: str | Path | None = None,
    verbose: bool = False,
) -> tuple[bool, dict]:
    """Validate one submitted solution_result-style JSON object in memory.

    Metric v2.0 behavior:
      - participant-submitted inputs are ignored;
      - official node/link/demand/setting files are loaded by case ID;
      - the submitted payload only needs an outputs section;
      - demand_multiplier is assigned from case_id;
      - validation and cost logic are aligned with fast_validator_v2_0.py.
    """
    if not isinstance(data, dict):
        raise ParticipantVisibleError("Submitted JSON payload must be an object/dict.")
    if "outputs" not in data:
        raise ParticipantVisibleError("Submitted JSON payload must contain an 'outputs' section.")

    nodes_df, links_df, demands_df, official_settings, official_input_dir = _load_official_inputs_for_case(case_id)
    settings = _settings_for_case_id(case_id)

    nodes_df = _df_with_columns(nodes_df.to_dict(orient="records"), [
        "node_id", "node_type", "name", "x_coord", "y_coord",
        "num_tracks", "handling_capacity", "handling_cost", "railroad_id",
    ])
    links_df = _df_with_columns(links_df.to_dict(orient="records"), [
        "link_id", "from_node_id", "to_node_id", "length", "capacity",
    ])
    demands_df = _df_with_columns(demands_df.to_dict(orient="records"), [
        "commodity_id", "commodity_type", "origin_yard_id", "dest_yard_id", "volume",
    ])

    demand_multiplier = float(settings.get("demand_multiplier", 1.0))
    if "volume" in demands_df.columns and demand_multiplier != 1.0:
        demands_df = demands_df.copy()
        demands_df["volume"] = demands_df["volume"].astype(float) * demand_multiplier

    outputs = data.get("outputs", {})
    blocks_df = _df_with_columns(outputs.get("1 Block Design", []), [
        "block_id", "from_yard_id", "to_yard_id", "block_type",
    ])
    seqs_df = _df_with_columns(outputs.get("2 Blocking Sequence", []), [
        "commodity_id", "commodity_type", "origin_yard_id", "dest_yard_id", "volume", "blocking_sequence",
    ])
    routes_df = _df_with_columns(outputs.get("3 Block Route", []), [
        "block_id", "from_yard_id", "to_yard_id", "physical_path_nodes", "physical_path_links",
    ])

    if "block_type" not in blocks_df.columns and "commodity_type" in blocks_df.columns:
        blocks_df["block_type"] = blocks_df["commodity_type"]

    res = {
        "checks": {},
        "shortest_path_engine": "pending",
        "od_matrix": {},
        "official_input_dir": str(official_input_dir),
        "settings_source": "official setting.csv if present; metric defaults otherwise; submitted inputs ignored",
        "demand_multiplier_source": "case_id",
    }
    is_valid = True

    # Index inputs. If the required ID columns are absent or empty, keep empty indexed frames.
    nodes_df = nodes_df.set_index("node_id", drop=True) if "node_id" in nodes_df.columns else nodes_df
    links_df = links_df.set_index("link_id", drop=True) if "link_id" in links_df.columns else links_df
    blocks_df = blocks_df.set_index("block_id", drop=True) if "block_id" in blocks_df.columns else blocks_df

    # ------- [1] Flow conservation -------
    seqs_df = seqs_df.copy()
    seq_split = seqs_df["blocking_sequence"].astype(str).str.split(" -> ") if "blocking_sequence" in seqs_df.columns else pd.Series([], dtype=object)
    seqs_df["bids"] = seq_split.apply(lambda L: [int(x) for x in L if str(x).strip()]) if len(seqs_df) else []
    seqs_df["first_bid"] = seqs_df["bids"].apply(lambda L: L[0] if L else -1) if len(seqs_df) else []
    seqs_df["last_bid"] = seqs_df["bids"].apply(lambda L: L[-1] if L else -1) if len(seqs_df) else []

    bf = blocks_df["from_yard_id"].astype(int).to_dict() if "from_yard_id" in blocks_df.columns else {}
    bt = blocks_df["to_yard_id"].astype(int).to_dict() if "to_yard_id" in blocks_df.columns else {}

    if len(seqs_df):
        seqs_df["seq_origin"] = seqs_df["first_bid"].map(bf)
        seqs_df["seq_dest"] = seqs_df["last_bid"].map(bt)
        bad_orig = seqs_df["seq_origin"].astype("Int64") != seqs_df["origin_yard_id"].astype("Int64")
        bad_dest = seqs_df["seq_dest"].astype("Int64") != seqs_df["dest_yard_id"].astype("Int64")
        n_endpoint_violations = int(bad_orig.sum() + bad_dest.sum())
    else:
        n_endpoint_violations = 0

    chain_violations = 0
    missing_block_refs = 0
    for _, r in seqs_df.iterrows():
        bids = r["bids"]
        if any(bid not in bf or bid not in bt for bid in bids):
            missing_block_refs += 1
            continue
        for i in range(len(bids) - 1):
            if bt.get(bids[i]) != bf.get(bids[i + 1]):
                chain_violations += 1
                break

    edge_set = set()
    if not links_df.empty:
        for _, l in links_df.iterrows():
            u, v = int(l["from_node_id"]), int(l["to_node_id"])
            edge_set.add((u, v)); edge_set.add((v, u))

    physical_od_mismatches = 0
    physical_edge_missing = 0
    route_block_missing = 0
    for _, route in routes_df.iterrows():
        try:
            bid = int(route["block_id"])
        except (KeyError, TypeError, ValueError):
            route_block_missing += 1
            continue
        if bid not in blocks_df.index:
            route_block_missing += 1
            continue
        node_path = _parse_arrow_ints(route.get("physical_path_nodes", ""))
        link_path_str = str(route.get("physical_path_links", "")).strip()
        # Match validator.py: skip empty path records instead of failing here.
        if not node_path or not link_path_str:
            continue
        if node_path[0] != int(blocks_df.loc[bid, "from_yard_id"]) or node_path[-1] != int(blocks_df.loc[bid, "to_yard_id"]):
            physical_od_mismatches += 1
        for u, v in zip(node_path[:-1], node_path[1:]):
            if (u, v) not in edge_set:
                physical_edge_missing += 1
                break

    n1_pass = (
        n_endpoint_violations == 0
        and chain_violations == 0
        and missing_block_refs == 0
        and physical_od_mismatches == 0
        and physical_edge_missing == 0
        and route_block_missing == 0
    )
    res["checks"]["1_flow_conservation"] = {
        "pass": n1_pass,
        "endpoint_mismatches": n_endpoint_violations,
        "chain_break_sequences": chain_violations,
        "missing_block_ref_sequences": missing_block_refs,
        "physical_od_mismatches": physical_od_mismatches,
        "physical_edge_missing": physical_edge_missing,
        "route_block_missing": route_block_missing,
    }
    if not n1_pass:
        is_valid = False

    # ------- [1](b) No-subtour / acyclic blocking sequence -------
    # A blocking sequence may not revisit a yard.
    # Example: A->B->C->B is a cycle/subtour and is invalid.
    n_subtour_violations = 0
    subtour_example = None
    for _, r in seqs_df.iterrows():
        bids = r.get("bids", [])
        if not bids:
            continue
        if any(bid not in bf or bid not in bt for bid in bids):
            continue  # already reported as missing/broken sequence in [1]
        visited_yards = [int(bf[bids[0]])]
        visited_yards.extend(int(bt[bid]) for bid in bids)
        if len(set(visited_yards)) != len(visited_yards):
            n_subtour_violations += 1
            if subtour_example is None:
                subtour_example = " -> ".join(str(y) for y in visited_yards)

    n1b_pass = (n_subtour_violations == 0)
    res["checks"]["1b_no_subtour"] = {
        "pass": n1b_pass,
        "n_violations": int(n_subtour_violations),
        "example_yard_sequence": subtour_example,
    }
    if not n1b_pass:
        is_valid = False

    # ------- [2] Yard track limits -------
    if len(blocks_df) and "block_type" in blocks_df.columns:
        class_types = {str(x).strip().lower() for x in CLASSIFICATION_BLOCK_TYPES}
        is_class = blocks_df["block_type"].astype(str).str.strip().str.lower().isin(class_types)
        class_blocks = blocks_df[is_class]
        out_per_yard = class_blocks.groupby("from_yard_id").size() if len(class_blocks) else pd.Series(dtype=float)
    else:
        out_per_yard = pd.Series(dtype=float)
    nt = nodes_df["num_tracks"].astype(float) if "num_tracks" in nodes_df.columns else pd.Series(dtype=float)
    overuse = (out_per_yard - nt.reindex(out_per_yard.index, fill_value=0)).clip(lower=0) if len(out_per_yard) else pd.Series(dtype=float)
    n_c2_violations = int((overuse > 0).sum()) if len(overuse) else 0
    n2_pass = (n_c2_violations == 0)
    res["checks"]["2_yard_track_limits"] = {
        "pass": n2_pass,
        "n_violations": n_c2_violations,
        "max_overuse": int(overuse.max()) if len(overuse) else 0,
    }
    if not n2_pass:
        is_valid = False

    # ------- [3] Yard handling capacity -------
    interm_records = []
    for _, r in seqs_df.iterrows():
        bids = r["bids"]
        vol = float(r["volume"])
        for bid in bids[1:]:
            interm_records.append((bf.get(bid), vol))
    if interm_records:
        idf = pd.DataFrame(interm_records, columns=["yard_id", "volume"]).dropna()
        handling_vol = idf.groupby("yard_id")["volume"].sum()
    else:
        handling_vol = pd.Series(dtype=float)

    hc = nodes_df["handling_capacity"].astype(float) if "handling_capacity" in nodes_df.columns else pd.Series(dtype=float)

    h_overuse = (handling_vol - hc.reindex(handling_vol.index, fill_value=0)).clip(lower=0) if len(handling_vol) else pd.Series(dtype=float)

    n_c3_violations = int((h_overuse > 0).sum()) if len(h_overuse) else 0
    n3_pass = (n_c3_violations == 0)
    res["checks"]["3_yard_handling_capacity"] = {
        "pass": n3_pass,
        "n_violations": n_c3_violations,
        "max_overuse_cars": float(h_overuse.max()) if len(h_overuse) else 0.0,
    }
    if not n3_pass:
        is_valid = False

    # ------- Actual flow and route distances shared by C4/C5/C6/cost -------
    rows = []
    for _, r in seqs_df.iterrows():
        vol = float(r["volume"])
        for bid in r["bids"]:
            rows.append((bid, vol))
    flow_df = pd.DataFrame(rows, columns=["block_id", "vol"]) if rows else pd.DataFrame(columns=["block_id", "vol"])
    actual_vol = flow_df.groupby("block_id")["vol"].sum() if len(flow_df) else pd.Series(dtype=float)

    link_len = links_df["length"].astype(float).to_dict() if "length" in links_df.columns else {}
    if len(routes_df) and "physical_path_links" in routes_df.columns:
        routes_df = routes_df.copy()
        routes_df["distance"] = routes_df["physical_path_links"].apply(lambda x: _distance_from_link_path(x, link_len))
        bid_dist = routes_df.set_index("block_id")["distance"].to_dict()
    else:
        bid_dist = {}

    used_bids = [int(b) for b, v in actual_vol.items() if float(v) > 0 and int(b) in blocks_df.index]
    od_pairs = {
        (int(blocks_df.loc[bid, "from_yard_id"]), int(blocks_df.loc[bid, "to_yard_id"]))
        for bid in used_bids
    }
    if len(demands_df) and {"origin_yard_id", "dest_yard_id"}.issubset(demands_df.columns):
        for _, d in demands_df.iterrows():
            try:
                od_pairs.add((int(d["origin_yard_id"]), int(d["dest_yard_id"])))
            except Exception:
                pass
    sp_cache, sp_engine, od_meta = _combined_shortest_path_cache(
        links_df=links_df,
        od_pairs=od_pairs,
        blocks_df=blocks_df,
        demands_df=demands_df,
        od_distance_matrix_path=od_distance_matrix_path,
        verbose=verbose,
    )
    res["shortest_path_engine"] = sp_engine
    res["od_matrix"] = od_meta

    # check the shortest path source
    if verbose:
        print("[OD MATRIX] path:", od_meta.get("od_matrix_path"))
        print("[OD MATRIX] found:", od_meta.get("od_matrix_found"))
        print("[OD MATRIX] loaded pairs:", od_meta.get("od_matrix_loaded_pairs"))
        print("[OD MATRIX] requested pairs:", od_meta.get("requested_shortest_path_pairs"))
        print("[OD MATRIX] from matrix:", od_meta.get("shortest_path_pairs_from_matrix"))
        print("[OD MATRIX] from scipy:", od_meta.get("shortest_path_pairs_from_scipy"))
        print("[OD MATRIX] engine:", sp_engine)

    # ------- [4] Minimum block volume -------
    min_under_amount = 0.0
    shortest_path_missing_c4 = 0
    n_c4_violations = 0
    for bid in used_bids:
        u = int(blocks_df.loc[bid, "from_yard_id"])
        v = int(blocks_df.loc[bid, "to_yard_id"])
        sp = sp_cache.get((u, v), np.nan)
        if pd.isna(sp) or not np.isfinite(sp) or sp <= 0:
            shortest_path_missing_c4 += 1
            continue
        req_vol = _min_vol_scalar(float(sp), settings)
        b_vol = float(actual_vol.get(bid, 0.0))
        if b_vol < req_vol:
            n_c4_violations += 1
            min_under_amount = max(min_under_amount, req_vol - b_vol)

    n4_pass = (n_c4_violations == 0 and shortest_path_missing_c4 == 0)
    res["checks"]["4_min_block_volume"] = {
        "pass": n4_pass,
        "n_violations": n_c4_violations,
        "shortest_path_missing": shortest_path_missing_c4,
        "max_under_amount": float(min_under_amount),
        "checked_used_blocks": len(used_bids),
    }
    if not n4_pass:
        is_valid = False

    # ------- [5] Link capacities -------
    link_flow = pd.Series(0.0, index=links_df.index, dtype=float) if len(links_df) else pd.Series(dtype=float)
    pairs = []
    for _, r in routes_df.iterrows():
        try:
            bid = int(r["block_id"])
        except Exception:
            continue
        s = r.get("physical_path_links", "")
        if not s:
            continue
        vol = float(actual_vol.get(bid, 0.0))
        if vol == 0:
            continue
        for lid in _parse_arrow_ints(s):
            pairs.append((int(lid), vol))
    if pairs:
        lp = pd.DataFrame(pairs, columns=["link_id", "vol"])
        link_flow = lp.groupby("link_id")["vol"].sum().reindex(links_df.index, fill_value=0)
    cap = links_df["capacity"].astype(float) if "capacity" in links_df.columns else pd.Series(dtype=float)
    util = (link_flow / cap.replace(0, np.nan)).fillna(0) if len(cap) else pd.Series(dtype=float)
    over = link_flow - cap if len(cap) else pd.Series(dtype=float)
    n_c5_violations = int((over > 0).sum()) if len(over) else 0
    n5_pass = (n_c5_violations == 0)
    res["checks"]["5_link_capacity"] = {
        "pass": n5_pass,
        "n_violations": n_c5_violations,
        "max_utilization": float(util.max()) if len(util) else 0.0,
    }
    if not n5_pass:
        is_valid = False

    # ------- [6] Max circuitous ratio -------
    max_ratio = 0.0
    n_c6_violations = 0
    shortest_path_missing_c6 = 0
    for bid in used_bids:
        actual_d = float(bid_dist.get(int(bid), 0.0))
        u = int(blocks_df.loc[bid, "from_yard_id"])
        v = int(blocks_df.loc[bid, "to_yard_id"])
        sp = sp_cache.get((u, v), np.nan)
        if pd.isna(sp) or not np.isfinite(sp) or sp <= 0:
            shortest_path_missing_c6 += 1
            continue
        ratio = actual_d / float(sp)
        max_ratio = max(max_ratio, ratio)
        if actual_d > float(sp) * float(settings["max_circuitous_ratio"]) + 1e-4:
            n_c6_violations += 1
    n6_pass = (n_c6_violations == 0 and shortest_path_missing_c6 == 0)
    res["checks"]["6_max_circuitous_ratio"] = {
        "pass": n6_pass,
        "n_violations": n_c6_violations,
        "shortest_path_missing": shortest_path_missing_c6,
        "max_ratio": round(max_ratio, 4),
    }
    if not n6_pass:
        is_valid = False

    # ------- [7] Single path uniqueness -------
    if "commodity_type" in seqs_df.columns and len(seqs_df):
        grp = seqs_df.groupby(["commodity_id", "commodity_type"])["blocking_sequence"].nunique()
    elif len(seqs_df):
        grp = seqs_df.groupby(["commodity_id"])["blocking_sequence"].nunique()
    else:
        grp = pd.Series(dtype=float)
    n_c7_violations = int((grp > 1).sum()) if len(grp) else 0
    n7_pass = (n_c7_violations == 0)
    res["checks"]["7_single_path_uniqueness"] = {
        "pass": n7_pass,
        "n_split_demands": n_c7_violations,
    }
    if not n7_pass:
        is_valid = False

    # ------- [8] Blocking rule: single commodity type per block -------
    block_type_seen: dict[int, str] = {}
    n_c8_violations = 0
    for _, r in seqs_df.iterrows():
        c_type = str(r.get("commodity_type", "merchandise")).strip().lower()
        for bid in r.get("bids", []):
            bid = int(bid)
            if bid not in blocks_df.index:
                continue
            if bid in block_type_seen and block_type_seen[bid] != c_type:
                n_c8_violations += 1
            else:
                block_type_seen[bid] = c_type
    n8_pass = (n_c8_violations == 0)
    res["checks"]["8_blocking_rule_single_commodity_type"] = {
        "pass": n8_pass,
        "n_mixed_block_type_violations": int(n_c8_violations),
    }
    if not n8_pass:
        is_valid = False

    # ------- [9] Direct-block rule for Intermodal / Automobile -------
    # These commodity types must use a single OD block.
    n_direct_block_violations = 0
    max_direct_only_blocks_used = 0
    direct_block_example = None
    for _, r in seqs_df.iterrows():
        c_type = str(r.get("commodity_type", ""))
        if not _is_direct_only(c_type):
            continue
        bids = r.get("bids", [])
        max_direct_only_blocks_used = max(max_direct_only_blocks_used, len(bids))
        if len(bids) > 1:
            n_direct_block_violations += 1
            if direct_block_example is None:
                direct_block_example = str(r.get("blocking_sequence", ""))

    n9_direct_pass = (n_direct_block_violations == 0)
    res["checks"]["9_direct_block_rule"] = {
        "pass": n9_direct_pass,
        "n_violations": int(n_direct_block_violations),
        "max_direct_only_blocks_used": int(max_direct_only_blocks_used),
        "example_blocking_sequence": direct_block_example,
    }
    if not n9_direct_pass:
        is_valid = False

    # ------- [9b] Demand transported-volume consistency -------
    # A demand is served only by the positive volume actually submitted in
    # "2 Blocking Sequence". Merely appearing in the output must not count as
    # full service. Unserved residual demand is handled by the stress penalty;
    # therefore under-service is not a validation error, but non-positive rows,
    # unknown demand references, OD/type mismatches, and over-service are errors.
    tol = 1e-6
    demand_key_to_info: dict[tuple[int, str], dict] = {}
    if len(demands_df):
        for _, d in demands_df.iterrows():
            try:
                key = (int(d.get("commodity_id", 0)), str(d.get("commodity_type", "")))
                demand_key_to_info[key] = {
                    "volume": float(d.get("volume", 0) or 0),
                    "origin_yard_id": int(d.get("origin_yard_id")),
                    "dest_yard_id": int(d.get("dest_yard_id")),
                }
            except Exception:
                continue

    submitted_volume_by_key: dict[tuple[int, str], float] = {}
    positive_submitted_volume_by_key: dict[tuple[int, str], float] = {}
    n_nonpositive_seq_volume = 0
    n_unknown_demand_refs = 0
    n_demand_od_mismatches = 0

    if len(seqs_df):
        for _, r in seqs_df.iterrows():
            try:
                key = (int(r.get("commodity_id", 0)), str(r.get("commodity_type", "")))
                vol = float(r.get("volume", 0) or 0)
            except Exception:
                n_unknown_demand_refs += 1
                continue

            submitted_volume_by_key[key] = submitted_volume_by_key.get(key, 0.0) + vol
            if vol > tol:
                positive_submitted_volume_by_key[key] = positive_submitted_volume_by_key.get(key, 0.0) + vol
            else:
                n_nonpositive_seq_volume += 1

            dem_info = demand_key_to_info.get(key)
            if dem_info is None:
                n_unknown_demand_refs += 1
                continue
            try:
                if (
                    int(r.get("origin_yard_id")) != int(dem_info["origin_yard_id"])
                    or int(r.get("dest_yard_id")) != int(dem_info["dest_yard_id"])
                ):
                    n_demand_od_mismatches += 1
            except Exception:
                n_demand_od_mismatches += 1

    n_overserved_demands = 0
    max_overserved_cars = 0.0
    for key, submitted_v in submitted_volume_by_key.items():
        required_v = float(demand_key_to_info.get(key, {}).get("volume", 0.0))
        over = submitted_v - required_v
        if over > tol:
            n_overserved_demands += 1
            max_overserved_cars = max(max_overserved_cars, over)

    n9b_pass = (
        n_nonpositive_seq_volume == 0
        and n_unknown_demand_refs == 0
        and n_demand_od_mismatches == 0
        and n_overserved_demands == 0
    )
    res["checks"]["9b_demand_volume_consistency"] = {
        "pass": n9b_pass,
        "nonpositive_sequence_volume_rows": int(n_nonpositive_seq_volume),
        "unknown_demand_references": int(n_unknown_demand_refs),
        "demand_od_mismatches": int(n_demand_od_mismatches),
        "overserved_demands": int(n_overserved_demands),
        "max_overserved_cars": float(max_overserved_cars),
        "note": "Under-served demand is allowed and penalized through stress-score unserved car-miles.",
    }
    if not n9b_pass:
        is_valid = False

    # ------- Cost calculation -------
    n_blocks = int(len(blocks_df))
    fixed_cost = n_blocks * float(settings["block_fixed_cost"])
    transport_cost = 0.0
    for bid in blocks_df.index:
        vol = float(actual_vol.get(int(bid), 0.0))
        dist = float(bid_dist.get(int(bid), 0.0))
        transport_cost += vol * dist * float(settings["transport_cost_coefficient"])
    hc_cost_map = nodes_df["handling_cost"].astype(float).to_dict() if "handling_cost" in nodes_df.columns else {}
    handling_cost = float(sum(handling_vol.get(yid, 0) * hc_cost_map.get(yid, 0) for yid in handling_vol.index))

    # v2.0 interchange cost: per-block two-endpoints definition.
    # For each used block, compare origin-yard railroad vs destination-yard railroad.
    # If both are recognized Class-I railroads and different, charge one interchange
    # for that block. Intermediate physical-path nodes are ignored.
    interchange_cost_per_car = float(settings.get("interchange_cost", 100.0))

    def _norm_rr(rr):
        try:
            if rr is None or pd.isna(rr):
                return ""
        except Exception:
            if rr is None:
                return ""
        s = str(rr).strip()
        if s in {"", "nan", "NaN", "None", "NONE", "null", "NULL"}:
            return ""
        try:
            x = float(s)
            if x == -1.0:
                return ""
            if x.is_integer():
                return str(int(x))
        except Exception:
            pass
        return "" if s == "-1" else s

    def _class_i_rr(rr):
        rr = _norm_rr(rr).upper()
        if rr == "CSXT":
            rr = "CSX"
        return rr if rr in CLASS_I_RAILROADS else ""

    rr_map = {}
    if "railroad_id" in nodes_df.columns:
        rr_map = {
            int(nid): _norm_rr(row.get("railroad_id", ""))
            for nid, row in nodes_df.iterrows()
        }

    interchange_cost = 0.0
    interchange_transition_count = 0

    if len(blocks_df):
        for bid, vol in actual_vol.items():
            try:
                bid = int(bid)
            except Exception:
                continue
            vol = float(vol)
            if vol <= 0 or bid not in blocks_df.index:
                continue

            origin_yard = int(blocks_df.loc[bid, "from_yard_id"])
            dest_yard = int(blocks_df.loc[bid, "to_yard_id"])
            origin_rr = _class_i_rr(rr_map.get(origin_yard, ""))
            dest_rr = _class_i_rr(rr_map.get(dest_yard, ""))

            if origin_rr and dest_rr and origin_rr != dest_rr:
                interchange_transition_count += 1
                interchange_cost += vol * interchange_cost_per_car

    total_cost = fixed_cost + transport_cost + handling_cost + interchange_cost

    res["cost"] = {
        "fixed": float(fixed_cost),
        "transport": float(transport_cost),
        "handling": float(handling_cost),
        "interchange": float(interchange_cost),
        "total": float(total_cost),
        "interchange_definition": "v2.0 per-block two-endpoints Class-I-only",
        "interchange_transition_count": int(interchange_transition_count),
        "interchange_class_i_set": sorted(CLASS_I_RAILROADS),
    }

    # ------- v2.0 stress metrics -------
    # Served demand is measured by actual submitted positive volume, not by whether
    # a (commodity_id, commodity_type) key appears in the output. This prevents
    # zero-volume pseudo-transportation rows from being counted as served demands.
    total_demand_vol = 0.0
    served_demand_vol = 0.0
    unserved_carmiles = 0.0
    M = float(settings.get("stress_penalty_M", 5.0))

    stress_sp_used = 0
    stress_shortest_path_missing = 0
    stress_zero_distance_used = 0
    missing_examples: list[tuple[int, int, int, float]] = []

    for _, d in demands_df.iterrows():
        vol = float(d.get("volume", 0) or 0)
        cid = int(d.get("commodity_id", 0))
        ct = str(d.get("commodity_type", ""))
        key = (cid, ct)
        actual_served = max(0.0, float(positive_submitted_volume_by_key.get(key, 0.0)))
        served_vol = min(actual_served, vol)
        unserved_vol = max(0.0, vol - served_vol)

        total_demand_vol += vol
        served_demand_vol += served_vol

        if unserved_vol > 0:
            try:
                o = int(d["origin_yard_id"])
                dst = int(d["dest_yard_id"])
                d_sp = sp_cache.get((o, dst), np.nan)

                if not _is_valid_sp_distance(o, dst, d_sp):
                    stress_shortest_path_missing += 1
                    if len(missing_examples) < 10:
                        missing_examples.append((cid, o, dst, unserved_vol))
                    continue

                if float(d_sp) == 0.0 and o == dst:
                    stress_zero_distance_used += 1

                stress_sp_used += 1
                unserved_carmiles += unserved_vol * float(d_sp)
            except Exception:
                stress_shortest_path_missing += 1

    if verbose:
        print("[STRESS DIST] shortest path used:", stress_sp_used)
        print("[STRESS DIST] shortest path missing:", stress_shortest_path_missing)
        print("[STRESS DIST] zero-distance same-yard OD used:", stress_zero_distance_used)

    if stress_shortest_path_missing > 0:
        # Do not silently compute a partial stress score. The rule is strictly:
        # OD matrix if available, otherwise SciPy shortest path on the physical network.
        examples = "; ".join(
            f"commodity={cid}, OD={o}->{dst}, volume={vol:g}"
            for cid, o, dst, vol in missing_examples
        )
        raise RuntimeError(
            f"Cannot compute stress score: {stress_shortest_path_missing} unserved demand rows "
            f"do not have a finite shortest-path distance from either the OD matrix or SciPy. "
            f"Examples: {examples}"
        )

    loaded_ratio = served_demand_vol / total_demand_vol if total_demand_vol > 0 else 1.0
    unserved_ratio = 1.0 - loaded_ratio
    stress_score = total_cost + M * unserved_carmiles
    res["stress_metrics"] = {
        "M_penalty_coefficient": M,
        "total_demand_cars": float(total_demand_vol),
        "served_demand_cars": float(served_demand_vol),
        "unserved_demand_cars": float(total_demand_vol - served_demand_vol),
        "loaded_demand_ratio": float(loaded_ratio),
        "unserved_demand_ratio": float(unserved_ratio),
        "unserved_carmiles": float(unserved_carmiles),
        "operating_cost": float(total_cost),
        "stress_score": float(stress_score),
        "shortest_path_distance_used_count": int(stress_sp_used),
        "shortest_path_missing_count": int(stress_shortest_path_missing),
        "zero_distance_same_yard_used_count": int(stress_zero_distance_used),
    }

    res["demand_multiplier"] = demand_multiplier
    res["is_valid"] = bool(is_valid)
    return bool(is_valid), res


def fast_validate_file(
    json_path: str | Path,
    case_id=None,
    od_distance_matrix_path: str | Path | None = None,
    verbose: bool = False,
) -> tuple[bool, dict]:
    """Convenience wrapper for local tests with official inputs selected by case ID."""
    if case_id is None:
        raise ParticipantVisibleError("fast_validate_file() requires case_id so official inputs can be selected.")
    json_path = Path(json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if od_distance_matrix_path is None:
        od_distance_matrix_path = json_path.parent / "od_distance_matrix.csv"
    return fast_validate_payload(
        data,
        case_id=case_id,
        od_distance_matrix_path=od_distance_matrix_path,
        verbose=verbose,
    )


## Step 4 — laddered multi-case `score()` using metric v2.0

This cell validates each non-empty submission row with `fast_validate_payload()` and computes the laddered score. Empty rows and invalid non-empty rows are both treated as unsolved scenarios. Valid rows contribute their v2.0 `stress_score` to the quality tie-breaker.

The v2.0 stress score uses operating cost plus the penalty on unserved car-miles, with shortest-path distances coming from the OD matrix first and SciPy sparse Dijkstra as fallback.


In [ ]:
# Laddered multi-case Kaggle-style scorer (v2.0 official-inputs build).
METRIC_FAST_VERSION = "2.0-official-inputs"
# This cell uses the validator functions defined above.

import re

# For local smoke testing in this notebook. In Kaggle, `score()` receives these
# DataFrames directly, so these file paths are only for manual testing.
# Change these names if your local CSV files use different filenames.
solution_csv_path = "solution.csv"
submission_csv_path = "submission.csv"

# Print one console-readable feasibility summary per scenario/case.
# Use plain print() output rather than notebook display(), so logs are visible
# in VS Code, Kaggle, and non-interactive runs.
PRINT_FEASIBILITY_SUMMARY = True
SUMMARY_SEPARATOR = "-" * 88

# Group-specific objective/penalty normalization denominators.
# Rows 0-1-2 = group 0, rows 3-4-5 = group 1, rows 6-7-8 = group 2.
# Local score-normalization calibration, 2026-07-02.
# After the refreshed demand files, L1 increased from the old small instance
# to about 3.12M cars, so the previous group-0 scale of 5B was no longer
# comparable with L2/L3. These denominators only normalize the local ladder
# quality tie-breaker; they do not change feasibility validation or solution JSON.
GROUP_QUALITY_SCALES = {
    0: 15_000_000_000,
    1: 18_000_000_000,
    2: 20_000_000_000,
}

DEFAULT_GROUP_QUALITY_SCALE = 20_000_000_000

# Scoring rule:
#   - Empty rows are unsolved and excluded from quality averages.
#   - Non-empty invalid rows are also treated as unsolved, like empty rows.
#   - Valid rows contribute the case-level STRESS SCORE.
# The ladder interval already penalizes unsolved rows through solved_scenarios.

# Keep final scores inside the selected ladder interval. This preserves the
# lexicographic meaning of the interval table: solving more cases/scenarios is
# always more important than the objective value tie-breaker.
# Example: interval 0-6 is capped at 5.99, interval 6-12 at 11.99.
CLIP_TO_INTERVAL_MAX_MINUS_MARGIN = True
INTERVAL_CAP_MARGIN = 0.01

# Ladder intervals: (solved_cases, solved_scenarios) -> (lower, upper).
# Lower score is better.
SCORE_INTERVALS = {
    (3, 9): (0.0, 6.0),
    (3, 8): (6.0, 12.0),
    (3, 7): (12.0, 18.0),
    (3, 6): (18.0, 24.0),
    (3, 5): (24.0, 30.0),
    (3, 4): (30.0, 36.0),
    (3, 3): (36.0, 42.0),
    (2, 6): (42.0, 48.0),
    (2, 5): (48.0, 54.0),
    (2, 4): (54.0, 60.0),
    (2, 3): (60.0, 66.0),
    (2, 2): (66.0, 72.0),
    (1, 3): (72.0, 81.0),
    (1, 2): (81.0, 90.0),
    (1, 1): (90.0, 99.0),
    (0, 0): (100.0, 100.0),
}


def _detect_submission_json_column(submission: pd.DataFrame, row_id_column_name: str) -> str:
    """Return the single non-ID submission column that stores the JSON payload."""
    candidates = [c for c in submission.columns if c != row_id_column_name]
    if len(candidates) != 1:
        raise ParticipantVisibleError(
            f"Submission must contain exactly one non-ID JSON column; found {candidates}."
        )
    return candidates[0]


def _is_empty_submission_cell(x) -> bool:
    """
    Treat blank / null / {} / [] cells as unsolved scenarios.

    These rows do not call the validator and do not enter the objective-quality
    average. They only affect the ladder interval through solved_scenarios.
    """
    if x is None:
        return True
    try:
        if pd.isna(x):
            return True
    except Exception:
        pass
    if isinstance(x, str):
        s = x.strip()
        if s == "":
            return True
        if s.lower() in {"nan", "none", "null", "na", "n/a"}:
            return True
        if s in {"{}", "[]"}:
            return True
        return False
    if isinstance(x, (dict, list, tuple, set)) and len(x) == 0:
        return True
    return False


def _money(x) -> str:
    """Format numeric values for console summaries."""
    try:
        if x is None or not np.isfinite(float(x)):
            return "n/a"
        return f"{float(x):,.0f}"
    except Exception:
        return "n/a"


def _print_case_feasibility_summary(
    case_id,
    position: int,
    group: int,
    status: str,
    validator_result: dict | None = None,
) -> None:
    """Print a plain-console feasibility summary for one scenario row."""
    if not PRINT_FEASIBILITY_SUMMARY:
        return

    print(SUMMARY_SEPARATOR)
    print(f"Case ID: {case_id} | position={position} | group={group} | status={status}")

    if validator_result is None:
        print("Feasibility check: skipped")
        print("Scoring treatment: unsolved / not counted in quality average")
        print(SUMMARY_SEPARATOR)
        return

    is_valid = bool(validator_result.get("is_valid", False))
    report = validator_result.get("validator_report", {}) or {}
    print(f"Feasibility check: {'PASS' if is_valid else 'FAIL'}")
    print(f"Shortest paths: {report.get('shortest_path_engine', 'n/a')}")

    checks = report.get("checks", {}) or {}
    for name, info in checks.items():
        passed = bool(info.get("pass", False))
        tag = "PASS" if passed else "FAIL"
        details = ", ".join(f"{k}={v}" for k, v in info.items() if k != "pass")
        print(f"  [{tag}] {name}: {details}")

    total_cost = validator_result.get("total_cost", np.nan)
    stress_score = validator_result.get("stress_score", np.nan)
    case_score = validator_result.get("case_score", np.nan)
    print(f"Operating cost:   {_money(total_cost)}")
    print(f"Stress score: {_money(stress_score)}")

    if is_valid:
        print(f"Case score:   {_money(case_score)}  (= stress score)")
        print("Scoring treatment: solved / counted in quality average")
    else:
        print("Case score:   n/a")
        print("Scoring treatment: invalid non-empty -> treated as unsolved, same as empty")
    print(SUMMARY_SEPARATOR)


def _validate_json_payload_for_score(payload: dict, case_id=None) -> dict:
    """Validate one non-empty scenario JSON using official inputs loaded by case ID."""
    if not isinstance(payload, dict):
        raise ParticipantVisibleError("Submitted JSON payload must be an object/dict.")
    if "outputs" not in payload:
        raise ParticipantVisibleError("Submitted JSON payload must contain an 'outputs' section.")

    # In metric v2.0, participant-submitted inputs are ignored.  The official
    # node/link/demand/setting CSV files are loaded inside fast_validate_payload().
    is_valid, report = fast_validate_payload(payload, case_id=case_id)
    total_cost = float(report["cost"]["total"])
    stress_score = float(report["stress_metrics"]["stress_score"])

    # Valid non-empty rows contribute the case-level Stress Score.
    # Invalid non-empty rows are treated like empty/unsolved rows and do not
    # contribute any quality value.
    case_score = stress_score if is_valid else np.nan

    return {
        "is_empty": False,
        "is_valid": bool(is_valid),
        "total_cost": total_cost,
        "stress_score": stress_score,
        "case_score": float(case_score) if np.isfinite(case_score) else np.nan,
        "quality_counted": bool(is_valid),
        "validator_report": report,
    }


def _select_interval(solved_cases: int, solved_scenarios: int) -> tuple[float, float]:
    """Return the ladder interval for the solved case/scenario count."""
    key = (int(solved_cases), int(solved_scenarios))
    if key not in SCORE_INTERVALS:
        raise ParticipantVisibleError(
            f"No score interval is defined for solved_cases={solved_cases}, "
            f"solved_scenarios={solved_scenarios}."
        )
    return SCORE_INTERVALS[key]


def _compute_ladder_score(case_report: pd.DataFrame) -> tuple[float, pd.DataFrame, dict]:
    """
    Convert per-scenario validation results into the final laddered score.

    Groups are defined by row position after sorting IDs:
        group 0 = rows 0,1,2
        group 1 = rows 3,4,5
        group 2 = rows 6,7,8

    Quality averaging rule:
        - Empty rows are unsolved and excluded from quality averages.
        - Non-empty invalid rows are also unsolved and excluded from quality averages.
        - Valid rows contribute the case-level Stress Score.
        - Group averages are computed only over valid rows in that group.
    """
    group_rows = []
    for g, df in case_report.groupby("group", sort=True):
        counted = df[df["quality_counted"]]
        group_score = float(counted["case_score"].mean()) if len(counted) else np.nan
        scale = float(GROUP_QUALITY_SCALES.get(int(g), DEFAULT_GROUP_QUALITY_SCALE))
        group_rows.append({
            "group": int(g),
            "ids": list(df["ID"]),
            "n_scenarios": int(len(df)),
            "n_empty": int(df["is_empty"].sum()),
            "n_nonempty": int((~df["is_empty"]).sum()),
            "n_valid": int(df["is_valid"].sum()),
            "n_invalid_nonempty": int(((~df["is_empty"]) & (~df["is_valid"])).sum()),
            "case_solved": bool(df["is_valid"].sum() > 0),
            "group_score": group_score,
            "quality_scale": scale,
            "group_quality_component": (group_score / scale) if np.isfinite(group_score) else np.nan,
            "avg_total_cost_valid_only": float(df.loc[df["is_valid"], "total_cost"].mean()) if df["is_valid"].any() else np.nan,
            "avg_stress_score_valid_only": float(df.loc[df["is_valid"], "stress_score"].mean()) if df["is_valid"].any() else np.nan,
            "avg_case_score_valid_only": group_score,
        })
    group_report = pd.DataFrame(group_rows)

    solved_cases = int(group_report["case_solved"].sum())
    solved_scenarios = int(case_report["is_valid"].sum())
    interval_low, interval_high = _select_interval(solved_cases, solved_scenarios)

    if solved_cases == 0 and solved_scenarios == 0:
        # The table fixes this at 100. Do not add a quality tie-breaker.
        quality_component_raw = 0.0
        quality_component_used = 0.0
        final_score = 100.0
        quality_clipped = False
    else:
        counted_group_quality = group_report["group_quality_component"].dropna()
        quality_component_raw = float(counted_group_quality.mean()) if len(counted_group_quality) else 0.0

        interval_width = interval_high - interval_low
        if CLIP_TO_INTERVAL_MAX_MINUS_MARGIN and interval_width > 0:
            max_quality = max(0.0, interval_width - INTERVAL_CAP_MARGIN)
            quality_component_used = min(max(0.0, quality_component_raw), max_quality)
        else:
            quality_component_used = quality_component_raw

        final_score = interval_low + quality_component_used
        quality_clipped = bool(abs(quality_component_used - quality_component_raw) > 1e-12)

    summary = {
        "solved_cases": solved_cases,
        "solved_scenarios": solved_scenarios,
        "interval_low": float(interval_low),
        "interval_high": float(interval_high),
        "interval_cap": float(interval_high - INTERVAL_CAP_MARGIN) if interval_high > interval_low else float(interval_high),
        "quality_component_raw": float(quality_component_raw),
        "quality_component_used": float(quality_component_used),
        "quality_clipped": bool(quality_clipped),
        "final_score": float(final_score),
        "empty_rows": int(case_report["is_empty"].sum()),
        "nonempty_rows": int((~case_report["is_empty"]).sum()),
        "invalid_nonempty_rows": int(((~case_report["is_empty"]) & (~case_report["is_valid"])).sum()),
        "group_quality_scales": dict(GROUP_QUALITY_SCALES),
        "valid_case_score_formula": "stress_score",
        "invalid_nonempty_treatment": "treated as unsolved / same as empty",
    }
    return float(final_score), group_report, summary


def score_details(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str = "ID") -> tuple[float, pd.DataFrame, pd.DataFrame, dict]:
    """
    Validate all scenario rows and compute the laddered score.

    Returns:
        final_score: final laddered metric value
        case_report: one row per ID/scenario
        group_report: one row per network group of three scenarios
        summary: solved counts, selected interval, and quality component
    """
    if row_id_column_name not in solution.columns:
        raise ParticipantVisibleError(f"solution is missing ID column {row_id_column_name!r}.")
    if row_id_column_name not in submission.columns:
        raise ParticipantVisibleError(f"submission is missing ID column {row_id_column_name!r}.")

    json_col = _detect_submission_json_column(submission, row_id_column_name)

    merged = solution[[row_id_column_name]].merge(
        submission[[row_id_column_name, json_col]],
        on=row_id_column_name,
        how="left",
        validate="one_to_one",
    )

    # Sort by numeric ID when possible; otherwise keep lexicographic ID order.
    try:
        merged = merged.assign(_id_sort=merged[row_id_column_name].astype(int)).sort_values("_id_sort")
    except Exception:
        merged = merged.sort_values(row_id_column_name).assign(_id_sort=np.arange(len(merged)))
    merged = merged.reset_index(drop=True)

    if len(merged) % 3 != 0:
        raise ParticipantVisibleError(
            f"Expected the number of rows to be a multiple of 3 scenarios per network; got {len(merged)}."
        )

    rows = []
    for pos, row in merged.iterrows():
        case_id = row[row_id_column_name]
        raw_cell = row[json_col]
        group = pos // 3

        if _is_empty_submission_cell(raw_cell):
            rows.append({
                "ID": case_id,
                "position": pos,
                "group": group,
                "is_empty": True,
                "is_valid": False,
                "total_cost": np.nan,
                "stress_score": np.nan,
                "case_score": np.nan,
                "quality_counted": False,
                "status": "empty_unsolved",
            })
            _print_case_feasibility_summary(case_id, pos, group, "empty_unsolved", None)
            continue

        try:
            payload = json.loads(raw_cell) if isinstance(raw_cell, str) else raw_cell
        except Exception as e:
            raise ParticipantVisibleError(f"ID {case_id}: invalid JSON in column {json_col!r}: {e}")

        # A parsed empty object/list is also treated as unsolved.
        if _is_empty_submission_cell(payload):
            rows.append({
                "ID": case_id,
                "position": pos,
                "group": group,
                "is_empty": True,
                "is_valid": False,
                "total_cost": np.nan,
                "stress_score": np.nan,
                "case_score": np.nan,
                "quality_counted": False,
                "status": "empty_unsolved",
            })
            _print_case_feasibility_summary(case_id, pos, group, "empty_unsolved", None)
            continue

        try:
            r = _validate_json_payload_for_score(payload, case_id=case_id)
        except ParticipantVisibleError:
            raise
        except Exception as e:
            raise ParticipantVisibleError(f"ID {case_id}: validator failed: {e}")

        status = "valid" if r["is_valid"] else "invalid_nonempty_treated_as_unsolved"
        rows.append({
            "ID": case_id,
            "position": pos,
            "group": group,
            "is_empty": False,
            "is_valid": r["is_valid"],
            "total_cost": r["total_cost"],
            "stress_score": r["stress_score"],
            "case_score": r["case_score"],
            "quality_counted": r["quality_counted"],
            "status": status,
        })
        _print_case_feasibility_summary(case_id, pos, group, status, r)

    case_report = pd.DataFrame(rows)
    final_score, group_report, summary = _compute_ladder_score(case_report)
    return final_score, case_report, group_report, summary


def score(solution: pd.DataFrame, submission: pd.DataFrame, row_id_column_name: str) -> float:
    """Kaggle entry point: return one finite float. Lower is better."""
    final_score, _, _, _ = score_details(solution, submission, row_id_column_name)
    if not np.isfinite(final_score):
        raise ParticipantVisibleError("Metric returned a non-finite score.")
    return float(final_score)


def _fmt_float(x, ndigits: int = 6) -> str:
    """Console-friendly numeric formatter for the compact global report."""
    try:
        if x is None or not np.isfinite(float(x)):
            return "n/a"
        return f"{float(x):,.{ndigits}f}"
    except Exception:
        return str(x)


def _print_compact_global_report(final_score: float, group_report: pd.DataFrame, summary: dict) -> None:
    """Print a compact global score report for console logs.

    The per-case feasibility details are printed earlier. This global report
    only shows the scale-level quality components and the final ladder formula.
    """
    sep = "-" * 88
    print("\n" + "=" * 88)
    print("GLOBAL SCORE REPORT")
    print("=" * 88)
    print("Per-scale quality components:")
    print(sep)
    print(f"{'scale':>5} | {'quality_scale':>16} | {'avg_case_score':>18} | {'avg_case_score / scale':>22}")
    print(sep)

    if group_report.empty:
        print("  n/a |              n/a |                n/a |                    n/a")
    else:
        for _, row in group_report.sort_values("group").iterrows():
            g = int(row["group"])
            quality_scale = row.get("quality_scale", np.nan)
            avg_case_score = row.get("avg_case_score_valid_only", np.nan)
            quality_component = row.get("group_quality_component", np.nan)
            print(
                f"{g:>5} | "
                f"{_fmt_float(quality_scale, 0):>16} | "
                f"{_fmt_float(avg_case_score, 2):>18} | "
                f"{_fmt_float(quality_component, 6):>22}"
            )
    print(sep)

    interval_low = float(summary.get("interval_low", np.nan))
    interval_high = float(summary.get("interval_high", np.nan))
    quality_component_raw = float(summary.get("quality_component_raw", np.nan))
    quality_component_used = float(summary.get("quality_component_used", np.nan))
    solved_cases = int(summary.get("solved_cases", 0))
    solved_scenarios = int(summary.get("solved_scenarios", 0))

    print(f"Solved cases/scenarios: {solved_cases} cases, {solved_scenarios} scenarios")
    print(f"Selected ladder interval: [{_fmt_float(interval_low, 2)}, {_fmt_float(interval_high, 2)})")
    print(f"Average scaled score: {_fmt_float(quality_component_raw, 6)}")
    if abs(quality_component_used - quality_component_raw) > 1e-12:
        print(f"Average scaled score used after interval cap: {_fmt_float(quality_component_used, 6)}")
    print()
    print("Final scoring formula:")
    print("  final_score = interval_lower_bound + average_scaled_score")
    print(f"  final_score = {_fmt_float(interval_low, 6)} + {_fmt_float(quality_component_used, 6)} = {_fmt_float(final_score, 6)}")
    print("=" * 88)


# ---- Local smoke test using local files, if present ----
if Path(solution_csv_path).exists() and Path(submission_csv_path).exists():
    solution_df = pd.read_csv(solution_csv_path)
    submission_df = pd.read_csv(submission_csv_path)

    final_score, case_report, group_report, score_summary = score_details(
        solution_df,
        submission_df,
        row_id_column_name="ID",
    )
    _print_compact_global_report(final_score, group_report, score_summary)
else:
    print(
        "Local smoke-test CSV files not found; score() is ready for Kaggle-style evaluation. "
        f"Expected: {solution_csv_path!r} and {submission_csv_path!r}."
    )
